<a href="https://colab.research.google.com/github/Solo7602/web/blob/3lab/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# 1. Загрузка датасета MovieLens (например, "ml-latest-small" для легкости)
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
import zipfile, io, requests
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

# 2. Создадим матрицу пользователь-товар (user-item)
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# 3. Вычислим косинусное сходство между товарами (фильмов)
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

# 4. Алгоритм рекомендаций для пользователя
def recommend_items(user_id, user_item_matrix, item_similarity_df, N=10):
    user_ratings = user_item_matrix.loc[user_id]
    high_rated_items = user_ratings[user_ratings >= 4].index  # товары с высокими оценками от пользователя

    scores = pd.Series(dtype=np.float64)
    for item in high_rated_items:
        sim_scores = item_similarity_df[item]
        scores = scores.add(sim_scores, fill_value=0)

    # Убираем товары, которые пользователь уже оценил
    scores = scores.drop(user_ratings[user_ratings > 0].index, errors='ignore')
    # Сортируем и берем топ-N
    recommendations = scores.sort_values(ascending=False).head(N)

    # Фильтрация для разнообразия (например, по жанрам)
    rec_movies = movies[movies['movieId'].isin(recommendations.index)]
    rec_movies = rec_movies.drop_duplicates(subset='genres')

    return rec_movies[['movieId', 'title', 'genres']].head(N)

# Пример рекомендации для пользователя с userId=n
recs = recommend_items(115, user_item_matrix, item_similarity_df, N=10)
print(recs)

# 5. Метрики Precision@N, Recall@N, MAP@N

def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    hits = sum(r in relevant_set for r in recommended_k)
    return hits / k

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    hits = sum(r in relevant_set for r in recommended_k)
    return hits / len(relevant_set) if relevant_set else 0

def average_precision(recommended, relevant, k):
    relevant_set = set(relevant)
    hits = 0
    sum_precisions = 0
    for i, r in enumerate(recommended[:k], start=1):
        if r in relevant_set:
            hits += 1
            sum_precisions += hits / i
    return sum_precisions / hits if hits > 0 else 0

# Разделим данные на train и test для оценки
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Построим матрицу train
train_matrix = train.pivot(index='userId', columns='movieId', values='rating').fillna(0)
item_sim_train = cosine_similarity(train_matrix.T)
item_sim_train_df = pd.DataFrame(item_sim_train, index=train_matrix.columns, columns=train_matrix.columns)

# Для каждого пользователя test получим рекомендации и сравним с реальными оценками
users_test = test['userId'].unique()
precisions, recalls, aps = [], [], []

for user in users_test:
    if user not in train_matrix.index:
        continue
    user_items_test = test[test['userId'] == user]
    relevant_items = user_items_test[user_items_test['rating'] >= 4]['movieId'].tolist()

    recs_for_user = recommend_items(user, train_matrix, item_sim_train_df, N=10)
    recommended_items = recs_for_user['movieId'].tolist()

    precisions.append(precision_at_k(recommended_items, relevant_items, 10))
    recalls.append(recall_at_k(recommended_items, relevant_items, 10))
    aps.append(average_precision(recommended_items, relevant_items, 10))

print(f'Precision@10: {np.mean(precisions):.3f}')
print(f'Recall@10: {np.mean(recalls):.3f}')
print(f'MAP@10: {np.mean(aps):.3f}')

/tmp/ipython-input-4067205789.py:15: PerformanceWarning: The following operation may generate 16966441536 cells in the resulting pandas object.
  user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)


KeyboardInterrupt: 